# Ejercicios Prácticos — Ingeniería de Datos con Pandas
### Basados en la Clase 4: Ingeniería de Datos con Pandas — Tema: E-commerce

Estos ejercicios usan **3 archivos** (`pedidos_online.csv`, `catalogo_productos_ecommerce.csv`, `clientes_ecommerce.json`) que simulan datos reales de una tienda online, con **problemas de calidad insertados intencionalmente** (nulos, duplicados, fechas malformadas) para que puedas practicar exactamente lo que viste en el notebook original: carga de datos, EDA, limpieza, `.dt`, `merge`, `groupby`/`agg`, `concat` y una consulta SQL con `sqlite3`.

Todo el contenido es generado y verificable por ti mismo ejecutando el código — no depende de fuentes externas, así que puedes comprobar cada resultado con `.shape`, `.isna().sum()`, `.dtypes`, etc.

> 📎 Antes de ejecutar, asegúrate de tener en la misma carpeta que este notebook: `pedidos_online.csv`, `catalogo_productos_ecommerce.csv`, `clientes_ecommerce.json`

## Estructura de los datos

**`pedidos_online.csv`** → `id_pedido, id_producto, cantidad, fecha, id_cliente, metodo_pago`
- 165 filas (160 originales + 5 duplicados exactos insertados, simulando doble clic en "Comprar")
- 21 valores nulos en `cantidad`
- Algunas fechas vienen con formato `AAAA/MM/DD` en lugar de `AAAA-MM-DD` (fechas inválidas para `pd.to_datetime`)

**`catalogo_productos_ecommerce.csv`** → `id_producto, producto, categoria, precio`
- 9 filas (8 productos + 1 duplicado exacto, simulando un error de sincronización del inventario)
- 1 valor nulo en `precio` (el de "Lámpara LED Escritorio")

**`clientes_ecommerce.json`** → `id_cliente, nombre, ciudad, cliente_premium`
- 20 clientes
- 1 valor nulo en `ciudad`

In [50]:
import pandas as pd
import numpy as np
import sqlite3

pd.set_option('display.max_columns', None)

---
## 🧩 Ejercicio 1 — Ingesta, EDA y Limpieza de Datos

**Objetivo:** practicar `read_csv`, `read_json`, `.info()`, `.describe()`, `isna()`, `fillna()`, `dropna()`, `duplicated()` y `drop_duplicates()`.

**Instrucciones:**

1. Carga los tres archivos (`pedidos_online.csv` con `parse_dates=['fecha']`, `catalogo_productos_ecommerce.csv`, `clientes_ecommerce.json` con `orient='records'`).
2. Haz una auditoría inicial de cada DataFrame con `.info()` y `.describe()`.
3. Cuenta los nulos por columna en cada tabla con `.isna().sum()`.
4. Trata los nulos con un criterio justificado (igual que en el notebook):
   - `cantidad` nulo en pedidos → imputa con `1` (mínimo lógico, se asume compra unitaria) y conviértelo a `int`.
   - `precio` nulo en catálogo → imputa con el precio promedio de su `categoria`.
   - `ciudad` nula en clientes → márcala como `'Ciudad Desconocida'`.
5. Detecta y elimina los duplicados:
   - Fila completamente duplicada en `catalogo_productos_ecommerce` (usa `duplicated(keep=False)` primero para verla, luego `drop_duplicates()`).
   - Duplicados exactos en `pedidos_online` (¿cuántas filas había antes y después?).
6. Convierte `fecha` con `pd.to_datetime(..., errors='coerce')`, identifica cuántas fechas quedaron como `NaT` y elimínalas con `dropna(subset=['fecha'])`.

**Pregúntate al final:** ¿cuántas filas tenías al inicio en `pedidos_online.csv` y cuántas te quedaron después de limpiar? Documenta cada decisión con un comentario, como pide la sección "Saber Ser" del notebook.

In [51]:
# 1. Carga de los archivos
df_pedidos = pd.read_csv('pedidos_online.csv', parse_dates=['fecha'])
df_productos = pd.read_csv('catalogo_productos_ecommerce.csv') 
df_clientes = pd.read_json('clientes_ecommerce.json', orient='records') 

print("\nArchivos Cargados Correctamente")
print(f"\nCantidad de Registros iniciales (Pedidos): {df_pedidos.shape}")
print(f"Cantidad de Registros iniciales (Productos): {df_productos.shape}")
print(f"Cantidad de Registros iniciales (Clientes): {df_clientes.shape}")



Archivos Cargados Correctamente

Cantidad de Registros iniciales (Pedidos): (165, 6)
Cantidad de Registros iniciales (Productos): (9, 4)
Cantidad de Registros iniciales (Clientes): (20, 4)


In [52]:
# 2. Auditoría inicial (.info() y .describe())
df_pedidos.info()
print()
df_productos.info()
print()
df_clientes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 165 entries, 0 to 164
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id_pedido    165 non-null    int64  
 1   id_producto  165 non-null    object 
 2   cantidad     144 non-null    float64
 3   fecha        165 non-null    object 
 4   id_cliente   165 non-null    object 
 5   metodo_pago  165 non-null    object 
dtypes: float64(1), int64(1), object(4)
memory usage: 7.9+ KB

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id_producto  9 non-null      object 
 1   producto     9 non-null      object 
 2   categoria    9 non-null      object 
 3   precio       8 non-null      float64
dtypes: float64(1), object(3)
memory usage: 420.0+ bytes

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (tot

In [53]:
df_pedidos.describe(include='all')

,id_pedido,id_producto,cantidad,fecha,id_cliente,metodo_pago
count,165.00000,165,144.000000,165,165,165
unique,NaN,8,NaN,107,20,5
top,NaN,SKU04,NaN,2025-02-12,CL13,Tarjeta de débito
freq,NaN,29,NaN,5,13,37
mean,79.69697,NaN,2.111111,NaN,NaN,NaN
std,46.66662,NaN,1.078143,NaN,NaN,NaN
min,1.00000,NaN,1.000000,NaN,NaN,NaN
25%,40.00000,NaN,1.000000,NaN,NaN,NaN
50%,79.00000,NaN,2.000000,NaN,NaN,NaN
75%,120.00000,NaN,3.000000,NaN,NaN,NaN


In [54]:
df_clientes.describe(include='all')

,id_cliente,nombre,ciudad,cliente_premium
count,20,20,19,20
unique,20,20,5,2
top,CL01,Ana Torres,Bogotá,False
freq,1,1,7,13


In [55]:
df_productos.describe(include='all')

,id_producto,producto,categoria,precio
count,9,9,9,8.000000
unique,8,8,3,NaN
top,SKU05,Teclado Mecánico,Electrónica,NaN
freq,2,2,7,NaN
mean,NaN,NaN,NaN,123000.000000
std,NaN,NaN,NaN,73546.875237
min,NaN,NaN,NaN,45000.000000
25%,NaN,NaN,NaN,62500.000000
50%,NaN,NaN,NaN,104500.000000
75%,NaN,NaN,NaN,180000.000000


In [56]:
# 3. Conteo de nulos por columna en cada tabla
df_clientes.isna().sum()

id_cliente         0
nombre             0
ciudad             1
cliente_premium    0
dtype: int64

In [57]:
df_pedidos.isna().sum()

id_pedido       0
id_producto     0
cantidad       21
fecha           0
id_cliente      0
metodo_pago     0
dtype: int64

In [58]:
df_productos.isna().sum()

id_producto    0
producto       0
categoria      0
precio         1
dtype: int64

In [59]:
# 4. Tratamiento de nulos
df_pedidos["cantidad"] = df_pedidos["cantidad"].fillna(1).astype(int)
df_productos["precio"] = df_productos["precio"].fillna(
    df_productos.groupby("categoria")["precio"].transform("mean"))
df_clientes["ciudad"] = df_clientes["ciudad"].fillna("Ciudad Desconocida")

In [60]:
# 5. Detección y eliminación de duplicados
df_clientes[df_clientes.duplicated()]

,id_cliente,nombre,ciudad,cliente_premium


In [61]:
#df_productos filtra el dataframe usando esa identificacion, duplicated(keep=False) se encarga de identificar
#las filas duplicadas y al final muestra el resultado de dichas filas duplicadas.
df_productos[df_productos.duplicated(keep=False)]

,id_producto,producto,categoria,precio
4,SKU05,Teclado Mecánico,Electrónica,180000.0
8,SKU05,Teclado Mecánico,Electrónica,180000.0


In [62]:
df_pedidos[df_pedidos.duplicated()]
print("Productos: ", df_productos.shape)

Productos:  (9, 4)


In [63]:
df_productos = df_productos.drop_duplicates()
df_pedidos = df_pedidos.drop_duplicates()

print("Productos: ", df_productos.shape)
print("Pedidos: ", df_pedidos.shape)

Productos:  (8, 4)
Pedidos:  (160, 6)


In [64]:
# 6. Conversión y limpieza de fechas
df_pedidos["fecha"] = pd.to_datetime(df_pedidos["fecha"], errors='coerce')

#Fechas inválidas 
print("Fechas Inválidas (Nat): ", df_pedidos["fecha"].isna().sum())

#Eliminacion de fechas inválidas
df_pedidos = df_pedidos.dropna(subset=["fecha"])

print(f"\nCantidad de Registros actuales (Pedidos): {df_pedidos.shape}")


Fechas Inválidas (Nat):  4

Cantidad de Registros actuales (Pedidos): (156, 6)


## Reflexión Final

En comparación con los datos iniciales referente a la tabla de los pedidos, al inicio la misma contaba con 165 filas en total. Luego de realizar limpieza el total de las filas correspondia a 156, por lo cual se puede concluir que solamente 9 registros fueron eliminados de la tabla de pedidos: 5 por ser duplicados exactos y 4 por tener un formato de fecha inválido.

---
## 🧩 Ejercicio 2 — Ingeniería de Fechas + GroupBy/Agg

**Objetivo:** practicar el acceso `.dt`, `groupby()` con `.agg()`, y `merge` (inner/left).

Usa el DataFrame de pedidos ya limpio del Ejercicio 1.

**Instrucciones:**

1. Extrae de la columna `fecha`: `anio`, `mes`, `nombre_mes`, `dia_semana` y `trimestre` (usa `.dt`).
2. Une `pedidos_online` con `catalogo_productos_ecommerce` (`merge`, `how='inner'`, llave `id_producto`) y luego con `clientes_ecommerce` (`how='left'`, llave `id_cliente`).
3. Crea la columna `total_pedido = cantidad * precio`.
4. Responde con `groupby` + `.agg()`:
   - Ingresos totales, número de pedidos y ticket promedio **por ciudad**.
   - Unidades vendidas e ingresos **por categoría y producto**, ordenado de mayor a menor ingreso.
   - Comparación de ingresos y ticket promedio entre `cliente_premium` (`True` vs `False`).
   - ¿Cuál es el `metodo_pago` más usado? ¿Y cuál genera más ingresos en promedio? (`groupby('metodo_pago')`)
   - ¿Qué día de la semana (`dia_semana`) recibe más pedidos? (`value_counts()`)

In [65]:
# 1. Ingeniería de fechas con .dt
df_pedidos['anio'] = df_pedidos['fecha'].dt.year
df_pedidos['mes'] = df_pedidos['fecha'].dt.month
df_pedidos['dia'] = df_pedidos['fecha'].dt.day
df_pedidos['mes_nombre'] = df_pedidos['fecha'].dt.month_name()
df_pedidos['dia_semana'] = df_pedidos['fecha'].dt.day_name()
df_pedidos['trimestre'] = df_pedidos['fecha'].dt.quarter

df_pedidos[['fecha', 'anio', 'mes', 'mes_nombre', 'dia', 'dia_semana', 'trimestre']].head(8)


,fecha,anio,mes,mes_nombre,dia,dia_semana,trimestre
0,2025-06-12,2025,6,June,12,Thursday,2
1,2025-05-10,2025,5,May,10,Saturday,2
2,2025-01-18,2025,1,January,18,Saturday,1
3,2025-04-15,2025,4,April,15,Tuesday,2
4,2025-06-27,2025,6,June,27,Friday,2
5,2025-02-01,2025,2,February,1,Saturday,1
6,2025-04-13,2025,4,April,13,Sunday,2
7,2025-01-20,2025,1,January,20,Monday,1


In [66]:
# 2. Merge: pedidos + catálogo (inner) + clientes (left)
df_merge = df_pedidos.merge(
    df_productos,
    on= "id_producto",
    how= "inner"
)

In [67]:
df_merge = df_merge.merge(
   df_clientes,
   on="id_cliente",
   how="left" 
)
print(df_merge.shape)
df_merge.head()

(156, 18)


,id_pedido,id_producto,cantidad,fecha,id_cliente,metodo_pago,anio,mes,dia,mes_nombre,dia_semana,trimestre,producto,categoria,precio,nombre,ciudad,cliente_premium
0,1,SKU03,4,2025-06-12,CL14,Tarjeta de débito,2025,6,12,June,Thursday,2,Smartwatch Fit,Electrónica,250000.0,Felipe Soto,Bogotá,False
1,2,SKU08,1,2025-05-10,CL17,Tarjeta de crédito,2025,5,10,May,Saturday,2,Parlante Portátil,Electrónica,120000.0,Valeria Ortiz,Bogotá,False
2,3,SKU01,4,2025-01-18,CL12,Tarjeta de débito,2025,1,18,January,Saturday,1,Audífonos Bluetooth,Electrónica,89000.0,Andrés Rojas,Barranquilla,False
3,4,SKU04,4,2025-04-15,CL02,Tarjeta de crédito,2025,4,15,April,Tuesday,2,Cargador Inalámbrico,Electrónica,45000.0,Luis Gómez,Medellín,True
4,5,SKU06,1,2025-06-27,CL18,PayPal,2025,6,27,June,Friday,2,Lámpara LED Escritorio,Hogar,NaN,Mateo Cano,Bogotá,False


In [68]:
# 3. Columna total_pedido = cantidad * precio
df_merge['total_pedido'] = df_merge['cantidad'] * df_merge['precio']

In [ ]:
# 4. Ingresos, número de pedidos y ticket promedio por ciudad
df_merge.groupby('ciudad').agg(
    ingresos_totales = ('total_pedido', 'sum'),
    numero_pedidos = ('total_pedido', 'count'),
    ticket_promedio = ('total_pedido', 'mean')
)

,ingresos_totales,numero_pedidos,ticket_promedio
ciudad,,,
Barranquilla,7606000.0,30,30
Bogotá,11231000.0,47,47
Cali,3709000.0,14,14
Cartagena,3280000.0,17,17
Ciudad Desconocida,1582000.0,7,7
Medellín,2996000.0,23,23


In [70]:
# 4. Unidades e ingresos por categoría y producto (ordenado desc)
df_merge.groupby(['categoria', 'producto']).agg(
    unidades_vendidas = ('cantidad','sum'),
    ingresos = ('total_pedido', 'sum')
).sort_values('ingresos', ascending=False)

unidades_vendidas    ingresos
categoria   producto                                             
Electrónica Smartwatch Fit                         43  10750000.0
            Parlante Portátil                      40   4800000.0
            Teclado Mecánico                       26   4680000.0
            Audífonos Bluetooth                    41   3649000.0
            Cargador Inalámbrico                   62   2790000.0
            Mouse Ergonómico                       36   1980000.0
Accesorios  Mochila Antirrobo                      27   1755000.0
Hogar       Lámpara LED Escritorio                 32         0.0

In [71]:
# 4. Comparación cliente_premium (True vs False)
df_merge.groupby('cliente_premium').agg(
    ingresos_totales = ('total_pedido', 'sum'),
    ticket_promedio = ('total_pedido', 'mean')
)

,ingresos_totales,ticket_promedio
cliente_premium,,
False,19401000.0,225593.023256
True,11003000.0,211596.153846


In [72]:
# 4. Método de pago más usado y el que más ingresos genera en promedio
metodo_pago_evaluacion = df_merge.groupby('metodo_pago').agg(
    numero_pedidos = ('total_pedido','count'),
    ingreso_promedio = ('total_pedido','mean')
)
metodo_pago_evaluacion.sort_values('numero_pedidos', ascending=False)
metodo_pago_evaluacion.sort_values('ingreso_promedio', ascending=False)

,numero_pedidos,ingreso_promedio
metodo_pago,,
Tarjeta de débito,31,251774.193548
Transferencia,25,233280.000000
PayPal,24,216375.000000
Contra entrega,29,210551.724138
Tarjeta de crédito,29,188551.724138


In [73]:
# 4. Día de la semana con más pedidos
df_merge.groupby('dia_semana')['id_pedido'].nunique()

dia_semana
Friday       21
Monday       31
Saturday     19
Sunday       25
Thursday     13
Tuesday      25
Wednesday    22
Name: id_pedido, dtype: int64

---
## 🧩 Ejercicio 3 — Concat, Auditoría con Outer Join y SQL desde Pandas

**Objetivo:** practicar `pd.concat()`, `merge(how='outer', indicator=True)` y `pd.read_sql()` con `sqlite3`.

**Instrucciones:**

1. Divide el DataFrame de pedidos limpio en dos mitades simulando dos centros de distribución (`Bodega Norte` y `Bodega Sur`, igual que en el notebook) y únelas de nuevo con `pd.concat(axis=0, ignore_index=True)`. Verifica que el total de filas cuadre.
2. Haz un `merge(how='outer', indicator=True)` entre `pedidos_online` y `catalogo_productos_ecommerce` por `id_producto`. Usa `value_counts()` sobre la columna `_merge` para auditar: ¿hay productos en el catálogo que nunca se vendieron? ¿hay pedidos con un `id_producto` que no existe en el catálogo?
3. Crea una conexión SQLite en memoria (`sqlite3.connect(':memory:')`), carga el DataFrame completo (unido con clientes) como tabla `pedidos_completo` usando `.to_sql()`, y con `pd.read_sql()` responde estas 2 consultas en SQL puro:
   - **Top 3 productos por ingresos totales** (`GROUP BY`, `SUM`, `ORDER BY`, `LIMIT`).
   - **Ingresos por mes, solo los meses con más de 10 pedidos** (`strftime('%Y-%m', fecha)`, `GROUP BY`, `HAVING`).
4. Cierra la conexión con `conn.close()`.

In [74]:
# 1. Split en dos "bodegas" y concat de nuevo
df_mitades = len(df_pedidos) // 2
bodega_del_norte = df_pedidos.iloc[:df_mitades].copy() #se guarda la mitad del df del 1 hasta la mitad
bodega_del_sur = df_pedidos.iloc[df_mitades:].copy() #se guarda desde la mitad restante del df hasta el final

bodega_del_norte['centro_distribucion']='Bodega Norte'
bodega_del_sur['centro_distribucion']='Bodega Sur'

bodega_del_norte.head()

,id_pedido,id_producto,cantidad,fecha,id_cliente,metodo_pago,anio,mes,dia,mes_nombre,dia_semana,trimestre,centro_distribucion
0,1,SKU03,4,2025-06-12,CL14,Tarjeta de débito,2025,6,12,June,Thursday,2,Bodega Norte
1,2,SKU08,1,2025-05-10,CL17,Tarjeta de crédito,2025,5,10,May,Saturday,2,Bodega Norte
2,3,SKU01,4,2025-01-18,CL12,Tarjeta de débito,2025,1,18,January,Saturday,1,Bodega Norte
3,4,SKU04,4,2025-04-15,CL02,Tarjeta de crédito,2025,4,15,April,Tuesday,2,Bodega Norte
4,5,SKU06,1,2025-06-27,CL18,PayPal,2025,6,27,June,Friday,2,Bodega Norte


In [75]:
bodega_del_sur.head()

,id_pedido,id_producto,cantidad,fecha,id_cliente,metodo_pago,anio,mes,dia,mes_nombre,dia_semana,trimestre,centro_distribucion
80,81,SKU03,1,2025-04-07,CL08,Tarjeta de crédito,2025,4,7,April,Monday,2,Bodega Sur
81,82,SKU01,2,2025-05-28,CL19,Transferencia,2025,5,28,May,Wednesday,2,Bodega Sur
82,83,SKU04,2,2025-05-22,CL16,Transferencia,2025,5,22,May,Thursday,2,Bodega Sur
83,84,SKU01,3,2025-02-03,CL01,Transferencia,2025,2,3,February,Monday,1,Bodega Sur
84,85,SKU06,1,2025-06-03,CL13,Tarjeta de crédito,2025,6,3,June,Tuesday,2,Bodega Sur


In [76]:
# 2. Merge outer con indicator=True para auditoría
df_union_pedidos = pd.concat([bodega_del_norte,bodega_del_sur], axis=0, ignore_index=True)
print("Bodega del Norte: ", bodega_del_norte.shape)
print("Bodega del Sur: ", bodega_del_sur.shape)
print("Daraframes Unidos: ", df_union_pedidos.shape)

assert len(df_union_pedidos) == len(df_pedidos), "Error!!!"

auditoria_pedidos = df_pedidos.merge(
    df_productos,
    on='id_producto',
    how='outer',
    indicator=True
)

auditoria_pedidos['_merge'].value_counts()

Bodega del Norte:  (78, 13)
Bodega del Sur:  (78, 13)
Daraframes Unidos:  (156, 13)


_merge
both          156
left_only       0
right_only      0
Name: count, dtype: int64

In [77]:
# 3. Conexión SQLite en memoria + carga de la tabla
conn = sqlite3.connect(':memory:')

# Se hace la carga de dataframes unidos para crear la tabla emn sqlite
df_merge.to_sql('pedidos_completos', conn, index=False, if_exists='replace')


156

In [78]:
# Verificando las tablas creadas en la base de datos
pd.read_sql(""" SELECT name FROM sqlite_master WHERE type = 'table'; """, conn)

,name
0,pedidos_completos


In [79]:
# Explorando la tabla pedidos_completos
query = """
SELECT * FROM pedidos_completos
"""
pd.read_sql(query, conn)


,id_pedido,id_producto,cantidad,fecha,id_cliente,metodo_pago,anio,mes,dia,mes_nombre,dia_semana,trimestre,producto,categoria,precio,nombre,ciudad,cliente_premium,total_pedido
0,1,SKU03,4,2025-06-12 00:00:00,CL14,Tarjeta de débito,2025,6,12,June,Thursday,2,Smartwatch Fit,Electrónica,250000.0,Felipe Soto,Bogotá,0,1000000.0
1,2,SKU08,1,2025-05-10 00:00:00,CL17,Tarjeta de crédito,2025,5,10,May,Saturday,2,Parlante Portátil,Electrónica,120000.0,Valeria Ortiz,Bogotá,0,120000.0
2,3,SKU01,4,2025-01-18 00:00:00,CL12,Tarjeta de débito,2025,1,18,January,Saturday,1,Audífonos Bluetooth,Electrónica,89000.0,Andrés Rojas,Barranquilla,0,356000.0
3,4,SKU04,4,2025-04-15 00:00:00,CL02,Tarjeta de crédito,2025,4,15,April,Tuesday,2,Cargador Inalámbrico,Electrónica,45000.0,Luis Gómez,Medellín,1,180000.0
4,5,SKU06,1,2025-06-27 00:00:00,CL18,PayPal,2025,6,27,June,Friday,2,Lámpara LED Escritorio,Hogar,NaN,Mateo Cano,Bogotá,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151,156,SKU07,1,2025-03-09 00:00:00,CL17,Tarjeta de débito,2025,3,9,March,Sunday,1,Mouse Ergonómico,Electrónica,55000.0,Valeria Ortiz,Bogotá,0,55000.0
152,157,SKU03,3,2025-06-17 00:00:00,CL04,Tarjeta de débito,2025,6,17,June,Tuesday,2,Smartwatch Fit,Electrónica,250000.0,Carlos Peña,Barranquilla,1,750000.0
153,158,SKU03,1,2025-03-22 00:00:00,CL19,Contra entrega,2025,3,22,March,Saturday,1,Smartwatch Fit,Electrónica,250000.0,Daniela Paz,Barranquilla,0,250000.0
154,159,SKU02,1,2025-06-22 00:00:00,CL02,Contra entrega,2025,6,22,June,Sunday,2,Mochila Antirrobo,Accesorios,65000.0,Luis Gómez,Medellín,1,65000.0


In [80]:
# 3a. Top 3 productos por ingresos totales (SQL)
query_2 = """
SELECT producto, SUM(total_pedido) AS Ingresos
FROM pedidos_completos
GROUP BY producto
ORDER BY Ingresos DESC
LIMIT 3
"""
pd.read_sql(query_2,conn)

,producto,Ingresos
0,Smartwatch Fit,10750000.0
1,Parlante Portátil,4800000.0
2,Teclado Mecánico,4680000.0


In [81]:
# 3b. Ingresos por mes, solo meses con más de 10 pedidos (SQL)
query_3 = """
    SELECT strftime('%Y-%m', fecha) AS mes, 
        COUNT(*) AS numero_pedidos,
        SUM(total_pedido) AS Ingresos
    FROM pedidos_completos
    GROUP BY mes
    HAVING numero_pedidos > 10
    ORDER BY mes
"""
pd.read_sql(query_3, conn)


,mes,numero_pedidos,Ingresos
0,2025-01,29,5735000.0
1,2025-02,30,4494000.0
2,2025-03,29,4054000.0
3,2025-04,24,6186000.0
4,2025-05,19,2774000.0
5,2025-06,25,7161000.0


In [82]:
# 4. Cerrar la conexión
conn.close()


---
## ✅ Criterios de verificación (para que revises tu propio trabajo)

- El `.shape` final de `pedidos_online` limpio debe ser menor al original (por duplicados y fechas inválidas eliminadas).
- `catalogo_productos_ecommerce` limpio debe tener 8 filas (una menos que el original, por el duplicado).
- Ninguna columna clave (`cantidad`, `precio`, `ciudad`) debe tener nulos después de la limpieza — confírmalo con `.isna().sum()`.
- La suma de `total_pedido` calculada con pandas debe coincidir con la suma de `ingresos` que te da la consulta SQL del Ejercicio 3 (son la misma información, calculada por dos caminos distintos — es una buena forma de verificar que no cometiste errores).

In [88]:
# Espacio libre para tu verificación final
print(df_pedidos.shape)
print(df_productos.shape)
print(df_productos.isna().sum())

(156, 12)
(8, 4)
id_producto    0
producto       0
categoria      0
precio         1
dtype: int64
